In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import re
import warnings
warnings.filterwarnings('ignore')

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

In [ ]:
df = pd.read_csv(r'data/IMDB Dataset.csv')

In [ ]:
print(f"Dataset shape: {df.shape}")
print(df.head(10))

print("Distribution des labels:")
print(df['sentiment'].value_counts())

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(token) for token in tokens 
              if token not in stop_words and len(token) > 2]
    
    return ' '.join(tokens)

In [ ]:
print("Application du prétraitement")
df['review_clean'] = df['review'].apply(preprocess_text)

In [ ]:
cv = CountVectorizer(max_features=5000, max_df=0.95, min_df=2, stop_words='english')
cv_matrix = cv.fit_transform(df['review_clean'])

print(f"Matrice document-terme shape: {cv_matrix.shape}")
print(f"Nombre de documents: {cv_matrix.shape[0]}")
print(f"Nombre de mots uniques: {cv_matrix.shape[1]}")

In [ ]:
lda_model = LatentDirichletAllocation(
    n_components=10,
    learning_method='online',
    random_state=42,
    max_iter=10,
    verbose=0
)

lda_model.fit(cv_matrix)

In [ ]:
def display_topics(model, feature_names, no_top_words=10):
    topics_dict = {}
    for topic_idx, topic in enumerate(model.components_):
        top_words_idx = topic.argsort()[-no_top_words:][::-1]
        top_words = [feature_names[i] for i in top_words_idx]
        top_weights = topic[top_words_idx]
        
        topics_dict[topic_idx] = {
            'words': top_words,
            'weights': top_weights
        }
        print(f"\nTopic {topic_idx}:")
        print(f"  Mots: {', '.join(top_words)}")
        print(f"  Poids: {top_weights.round(3)}")
    return topics_dict

feature_names = cv.get_feature_names_out()
print("Topics extraits par LDA:")
lda_topics = display_topics(lda_model, feature_names, no_top_words=10)

In [ ]:
interpretations = {
    0: "Émotions positives (great, good, best)",
    1: "Critique négative (bad, worst, terrible)",
    2: "Acteurs et performance (actor, acting, character)",
    3: "Expérience cinématographique (watch, time, see)",
    4: "Qualité du film (movie, film, picture)",
    5: "Divertissement (fun, enjoy, entertainment)",
    6: "Trame narrative (story, plot, end)",
    7: "Sensations (recommend, love, hate)",
    8: "Aspects techniques (camera, scene, effect)",
    9: "Jugement global (excellent, poor, rating)"
}

for topic_id, interpretation in interpretations.items():
    print(f"  Topic {topic_id}: {interpretation}")

In [ ]:
print("Évaluation du modèle LDA:")
perplexity = lda_model.perplexity(cv_matrix)
print(f"  Perplexité sur l'ensemble d'entraînement: {perplexity:.4f}")
print(f"  (Plus bas = mieux. Valeur typique: 20-100)")

In [ ]:
print("Vectorisation avec TfidfVectorizer")
tfidf = TfidfVectorizer(max_features=5000, max_df=0.95, min_df=2, stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['review_clean'])
print(f"Matrice TF-IDF shape: {tfidf_matrix.shape}")

In [ ]:
print("Entraînement du modèle NMF")
nmf_model = NMF(
    n_components=10,
    random_state=42,
    max_iter=500,
    init='nndsvd'
)

nmf_model.fit(tfidf_matrix)
print("Modèle NMF entraîné")

In [ ]:
print("Topics extraits par NMF:")
tfidf_feature_names = tfidf.get_feature_names_out()
nmf_topics = display_topics(nmf_model, tfidf_feature_names, no_top_words=10)

### Comparaison qualitative : LDA vs NMF

### LDA

**Forces :**

* Modèle probabiliste
* Chaque document peut contenir plusieurs topics
* Gère l'incertitude
* Plus flexible

**Faiblesses :**

* Plus lent
* Sensible aux paramètres
* Parfois moins facile à interpréter

### NMF

**Forces :**

* Simple et rapide
* Topics souvent plus faciles à interpréter
* Fonctionne bien avec TF-IDF

**Faiblesses :**

* Pas de probabilités
* Ne gère pas directement l'incertitude
* Pas un modèle génératif

### Conclusion

Sur notre corpus, **NMF semble plus facile à interpréter** car les topics sont plus distincts.

---

## Différence fondamentale entre LDA et NMF

### LDA

LDA est un **modèle probabiliste**.

Il considère qu'un document peut contenir plusieurs topics.

Exemple :

> Un avis peut être composé de 70% de topic "film" et 30% de topic "acteur".

### NMF

NMF est une **méthode de factorisation de matrices**.

Elle décompose les données en plusieurs topics et donne les mots les plus importants pour chaque topic.

### Quand utiliser lequel ?

* **LDA** → si on veut utiliser les probabilités et analyser la distribution des topics.
* **NMF** → si on veut des topics simples, rapides et faciles à interpréter.

**Dans notre cas, NMF est intéressant pour interpréter les thèmes des critiques de films.**


In [ ]:
!pip install gensim

import gensim
from gensim.corpora import Dictionary
from gensim.models import LdaModel

print("Conversion en format Gensim")
texts = [text.split() for text in df['review_clean']]

print("Créer dictionnaire et filtrer extrêmes")
dictionary = Dictionary(texts)
print(f"Dictionnaire initial: {len(dictionary)} mots")
dictionary.filter_extremes(no_below=5, no_above=0.7)
print(f"Dictionnaire après filtrage: {len(dictionary)} mots")
corpus = [dictionary.doc2bow(text) for text in texts]

print("Entraîner modèle LDA Gensim")
lda_gensim = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=10,
    random_state=42,
    passes=5,
    per_word_topics=True
)

print("Topics Gensim:")
for idx, topic in lda_gensim.print_topics(-1):
    print(f"Topic {idx}: {topic}")

### Comparaison : Scikit-learn vs Gensim

* Les résultats sont **similaires**, mais peuvent être légèrement différents.
* **Gensim** utilise un apprentissage par plusieurs passages sur les données (*online learning*).
* **Scikit-learn** utilise principalement le **batch learning**.
* **Gensim** offre plus de flexibilité pour l'inférence et la sauvegarde des modèles.
* Le **prétraitement des textes** influence fortement les résultats.

### Conclusion

Les deux bibliothèques permettent de faire du **topic modeling**, mais **Gensim est plus flexible**, tandis que **Scikit-learn est plus simple à utiliser**.
